# 🏎️ Laboratorio di Reinforcement Learning: Missione Hill Climb Racing
### Progettazione della Funzione di Reward (Reward Shaping su Pendenze Collinari)

Benvenuti nel laboratorio interattivo di Reinforcement Learning applicato al celebre gioco **Hill Climb Racing**!
In questa esperienza non programmiamo le azioni manuali della vettura, ma **progettiamo la funzione matematica di Reward** per guidare un algoritmo di Deep Reinforcement Learning (**PPO - Proximal Policy Optimization**) a guidare in modo autonomo, veloce e sicuro su tracciati generati proceduralmente (Perlin Noise).

---
### 🎯 L'Obiettivo Didattico
Portare la vettura oltre gli **800 metri di distanza** su colline ripide e sconnesse, evitando:
1. 💥 **Lo schianto fatale del pilota**: la testa non deve mai impattare con il suolo (cappottamento).
2. ⏳ **Il blocco in salita (timeout)**: perdere slancio nel fondo valle e restare fermi senza trazione.
3. 🚀 **L'airtime incontrollato**: decollare sui dossi senza controllo dell'angolo di atterraggio.


## 1. L'idea: imparare per tentativi ed errori

Nel **Reinforcement Learning** l'agente osserva lo stato della fisica ad ogni istante ($s_t$), sceglie un'azione ($a_t$: accelerare, frenare o veleggiare) e riceve una ricompensa numerica ($r_t$).
L'obiettivo dell'algoritmo PPO è massimizzare la somma dei premi futuri:
$$\max_{\pi} \mathbb{E} \left[ \sum_{t=0}^T \gamma^t r_t \right]$$

Il tuo compito è definire $r_t$ in modo che guidare veloce e mantenere le ruote a terra sia conveniente, mentre ribaltarsi risulti matematicamente catastrofico!


### 🛠️ Installazione e preparazione dell'ambiente di simulazione
Esegui la cella seguente per caricare il pacchetto didattico `play2learn_hillclimb`.


In [ ]:
# 1. Installazione pacchetti di simulazione e rendering su Google Colab
!apt-get update -qq
!apt-get install -y -qq swig
!pip install -q "gymnasium[box2d]" stable-baselines3 imageio imageio-ffmpeg tqdm rich matplotlib pygame
print("✅ Installazione completata!")


In [ ]:
import sys
from pathlib import Path
if '.' not in sys.path:
    sys.path.insert(0, '.')

import play2learn_hillclimb
print("✅ Modulo play2learn_hillclimb caricato con successo!")


---
## 2. Conosci la Fisica della Scena e il Veicolo

Prima di impostare le formule matematiche, analizziamo i sensori del veicolo:
* **Chassis (Telaio)**: collegato al pilota con giunti elastici e sospensioni a molla.
* **Due Ruote Motrici (Posteriore & Anteriore)**: forniscono trazione solo quando toccano il terreno (`on_ground = 1`).
* **Testa del Pilota**: sensore critico. Se tocca il suolo, l'episodio termina con uno schianto irreversibile!


In [ ]:
play2learn_hillclimb.mostra_mappa_ambiente()


---
## 3. Sensori (`obs`), Motori (`action`) ed Eventi di Fine Corsa

Ad ogni step ($0{,}016\text{ s}$ / 60 FPS), la tua funzione `custom_reward(obs, action, is_success, is_crashed, is_timeout)` riceve:

| Indice | Sensore Fisico | Significato & Unità di Misura |
| :---: | :--- | :--- |
| `obs[0]` | **Distanza X** | Avanzamento orizzontale in metri ($0 \to 1000\text{ m}$). Target: $\ge 800\text{ m}$. |
| `obs[1]` | **Quota Y** | Altezza della vettura rispetto al piano di spawn. |
| `obs[2]` | **Velocità VX** | Velocità orizzontale in m/s ($> 0$ verso destra, avanzamento). |
| `obs[3]` | **Velocità VY** | Velocità verticale in m/s ($> 0$ caduta verso il basso). |
| `obs[4]` | **Inclinazione Telaio** | Angolo di rollio in radianti ($0 = \text{orizzontale}$, $\pm\pi$). Inclinazione $> 0.8\text{ rad}$ indica impennata o rischio ribaltamento. |
| `obs[5]` | **Velocità Angolare** | Velocità di rotazione del veicolo (rad/s). |
| `obs[6]` | **Ruota Posteriore a Terra** | $1.0$ se la ruota posteriore tocca l'erba, $0.0$ se è in aria. |
| `obs[7]` | **Ruota Anteriore a Terra** | $1.0$ se la ruota anteriore tocca l'erba, $0.0$ se è in aria. |

### Azioni Disponibili (`action`):
* `0`: **Motore Spento (Idle)** — Il veicolo veleggia per inerzia.
* `1`: **Acceleratore (Gas)** — Trazione in avanti a piena coppia.
* `2`: **Freno / Retromarcia (Brake)** — Frena e contrasta le impennate violente.

### Eventi Booleani di Fine Corsa:
* `is_success`: `True` se l'auto supera gli **800 metri** di tracciato.
* `is_crashed`: `True` se la testa del pilota ha toccato terra (ribaltamento fatale).
* `is_timeout`: `True` se il veicolo resta bloccato nel fondo valle senza progresso.


### 🔍 Esplorazione: guarda i numeri in tempo reale
Esegui la cella seguente per simulare 10 step con acceleratore attivo e osservare la telemetria.


In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("HillClimbRacing-v0")
obs, _ = env.reset(seed=42)

nomi_azioni = {0: "0: Idle (Spento)  ", 1: "1: Gas (Avanti)   ", 2: "2: Freno (Brake)  "}

print("=" * 82)
print(" STEP | AZIONE             | DIST (m) | VX (m/s) | INCLINAZIONE | RUOTA POST | RUOTA ANT")
print("=" * 82)

for step in range(1, 11):
    action = 1  # Diamo gas!
    obs, reward, term, trunc, info = env.step(action)
    print(f" {step:4d} | {nomi_azioni[action]} | {obs[0]:7.2f}m | {obs[2]:6.2f}   | {obs[4]:10.2f} rad | {obs[6]:10.1f} | {obs[7]:9.1f}")

env.close()
print("=" * 82)


---
## 4. La Matematica dei Premi e le Trappole Classiche del Pilota 🧠

Quando si allena una vettura di Hill Climb Racing con RL, si incontrano tipicamente due estremi:

1. ⚠️ **La Trappola del 'Full Gas' (Ribaltamento sui Dossi)**:
   Se premi solo l'avanzamento (`dist`), l'agente darà sempre gas al 100%. Quando salta su una cresta, il veicolo vola in aria e la reazione di coppia del motore fa ruotare il telaio all'indietro: il pilota atterra sulla testa e muore!
   *💡 Soluzione*: premiare la velocità solo quando almeno una ruota tocca terra (`obs[6] > 0.5 or obs[7] > 0.5`) e penalizzare inclinazioni eccessive in volo (`abs(obs[4]) > 0.5`).

2. ⚠️ **La Trappola della 'Prudenza Eccessiva' (Blocco nei Fossi)**:
   Se penalizzi troppo il ribaltamento, l'agente impara che fermarsi a 20 metri da terra è meno rischioso che saltare. Ma così non prenderà mai la rincorsa necessaria per risalire la collina successiva!
   *💡 Soluzione*: dare un bonus crescente proporzionale all'avanzamento della distanza (`obs[0] * 0.1`) e un premio consistente per il superamento dell'obiettivo (`is_success`).


In [ ]:
def custom_reward(obs, action, is_success, is_crashed, is_timeout):
    """
    Progetta qui la tua funzione matematica di Reward!

    SENSORI:
      dist_x   = obs[0]  # Distanza percorsa in metri (0 -> 1000)
      quota_y  = obs[1]  # Altezza del veicolo
      vx       = obs[2]  # Velocità orizzontale (> 0 = avanti)
      vy       = obs[3]  # Velocità verticale
      angle    = obs[4]  # Inclinazione telaio in radianti
      ang_vel  = obs[5]  # Velocità angolare di rotazione
      rear_on  = obs[6]  # Ruota posteriore a contatto (1.0 o 0.0)
      front_on = obs[7]  # Ruota anteriore a contatto (1.0 o 0.0)

    ESITI TERMINALI:
      is_success : True se raggiungi almeno 800m
      is_crashed : True se il pilota urta la testa (cappottamento)
      is_timeout : True se resti bloccato senza slancio
    """
    # 1. Eventi catastrofici e vittoria finale
    if is_crashed:
        return -100.0
    if is_timeout:
        return -40.0
    if is_success:
        return 120.0

    reward = 0.0

    # 2. Premio per avanzamento e velocità con trazione
    vx = float(obs[2])
    dist_x = float(obs[0])
    has_traction = (obs[6] > 0.5 or obs[7] > 0.5)

    if has_traction:
        # Premia la spinta in avanti quando le ruote toccano il suolo
        reward += max(0.0, vx * 1.5)
    else:
        # Penalità lieve per perdita prolungata di trazione
        reward -= 0.1

    # Piccolo incentivo progressivo per la distanza raggiunta
    reward += dist_x * 0.02

    # 3. Penalità per inclinazione pericolosa del telaio (anti-ribaltamento)
    angle = abs(float(obs[4]))
    if angle > 0.5:
        reward -= (angle - 0.5) * 4.0

    # Penalità lieve per retromarcia ingiustificata
    if vx < -0.2:
        reward -= 0.5

    return float(reward)


---
## 5. Addestramento PPO e Telemetria di Guida

Avvia l'addestramento dell'agente PPO con la tua funzione `custom_reward`.
Di default impostiamo **150.000 step**: i primi tentativi superano i primi 100-200 metri, mentre verso i 100k-150k step il pilota apprende a dosare il gas sui dossi per mantenere l'assetto piatto e superare gli 800 metri!


In [ ]:
# Avvia l'addestramento PPO su 4 ambienti paralleli
risultati = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="campagna_standard",
    total_timesteps=150_000,
    crea_gif_evoluzione=True,
    seed=42,
)


---
## 6. 🏆 Livelli Sfida Avanzati: Metti alla Prova la Vettura su Tracciati Estremi!

Hai superato il livello base con oltre l'80% di completamento?
Scegli una delle **Missioni Speciali** con pendenze e fisiche differenti:
* `"deserto_dune"`: Dune ripide, forte attrito e rischio insabbiamento.
* `"montagna_estrema"`: Pendenze rocciose quasi verticali, richiede perfetto controllo del freno.
* `"luna_bassa_gravita"`: Gravità 2.5 m/s²: salti altissimi e reazione di coppia accentuata.
* `"autostrada_veloce"`: Time trial di pura velocità su saliscendi dolci.


In [ ]:
# Scegli la missione tra: "deserto_dune", "montagna_estrema", "luna_bassa_gravita", "autostrada_veloce"
risultati_sfida = play2learn_hillclimb.run_lab(
    reward_fn=custom_reward,
    mission="deserto_dune",
    total_timesteps=150_000,
    seed=101,
)


---
## 7. 📊 Reportistica, GIF Timelapse e Preferiti Personali


In [ ]:
# 1. Salva la run completata nei tuoi preferiti personali
risultati.preferito(
    nome="Assetto_Colline_Stabile",
    note="Formula anti-ribaltamento calibrata: ottima trazione nelle valli ripide."
)


In [ ]:
# 2. Mostra la galleria di tutti i tuoi preferiti
play2learn_hillclimb.mostra_preferiti()


In [ ]:
# 3. Genera la Scheda Tecnica dell'esperimento (PDF vettoriale, HTML e Markdown)
report = risultati.report(
    autore="Studente",
    titolo="Scheda Tecnica Hill Climb - Missione Campagna"
)


In [ ]:
# 4. Metti a confronto due o più run per analizzare l'impatto dei parametri
play2learn_hillclimb.confronta_run([risultati, risultati_sfida])


---
## 8. 🕒 Reportistica da Timestamp (Recupera Run Passate)

Puoi recuperare qualsiasi run effettuata inserendo il suo timestamp:


In [ ]:
# Inserisci il timestamp desiderato oppure 'latest' per l'ultima run
timestamp_scelto = "latest"

report_recuperato = play2learn_hillclimb.genera_report_da_timestamp(
    timestamp_or_id=timestamp_scelto,
    autore="Studente",
)


### 📈 Confronto Diretto tra Run Passate (tramite Timestamp)

Vuoi mettere a confronto due versioni della tua vettura per analizzare chi va più lontano?  
Inserisci i due timestamp nelle due celle seguenti per avviare il confronto testa a testa con **valutazione estesa su 20 gare** e **video split-screen affiancati**!


In [ ]:
# @title 🕒 1. Inserisci il Timestamp del Primo Esperimento (Run A) { run: "auto" }
# Inserisci il timestamp del primo esperimento (es. '20261116_101530' oppure 'latest'):
timestamp_1 = "latest"  # @param {type:"string"}
print(f"✅ Primo esperimento impostato: {timestamp_1}")


In [ ]:
# @title 📊 2. Inserisci il Timestamp del Secondo Esperimento (Run B) e Avvia il Confronto { run: "auto" }
# Inserisci il timestamp del secondo esperimento (es. '20261116_094520' oppure 'previous'):
timestamp_2 = "previous"  # @param {type:"string"}

# Unione automatica dei timestamp e avvio della valutazione estesa su 20 gare con video split-screen
timestamp_confronto = f"{timestamp_1.strip()}, {timestamp_2.strip()}"
play2learn_hillclimb.confronta_run(timestamp_confronto)


### 📦 Scarica l'Archivio degli Esperimenti (ZIP per Google Colab)

Vuoi scaricare sul tuo computer tutti i modelli addestrati, i grafici di telemetria, i report PDF e i video generati durante la sessione?  
Esegui la cella seguente per comprimere la cartella degli esperimenti in un file compresso `runs_hillclimb.zip` pronto per il download:


In [ ]:
# Comprime l'intera cartella delle run e avvia automaticamente il download su Google Colab
play2learn_hillclimb.comprimi_runs(nome_zip="runs_hillclimb.zip")


---
## 9. 🎮 Sfida Interattiva: Pilota Umano vs Agente AI (e Single Player)

Pensi sia facile guidare su queste colline senza cappottare? Mettiti alla prova direttamente nel notebook!


### 9.1 🚀 Modalità Single Player (Guida Manuale)
Guida la vettura con i tasti freccia o touch a schermo:
* ⬆️ / W = Gas
* ⬇️ / S = Freno e Retromarcia
* ⬅️ ➡️ = Bilanciamento rollio in aria


In [ ]:
# @title 🏎️ Simulatore Arcade Single Player { run: "auto" }
# @markdown Scegli il tracciato su cui guidare:
scenario = "campagna_standard"  # @param ["campagna_standard", "deserto_dune", "montagna_estrema", "luna_bassa_gravita", "autostrada_veloce"]

# Avvia il simulatore con seed casuale ogni volta! (Oppure specifica es. seed=42 per un tracciato fisso)
play2learn_hillclimb.gioca_single_player(mission=scenario)


### 9.2 ⚔️ Sfida Uomo vs Agente AI (Split-Screen)
Gara in tempo reale sullo stesso tracciato collinare:
**Auto Rossa (Tu)** contro **Auto Verde (Il tuo Agente PPO)**! Chi arriverà più lontano?


In [ ]:
# @title ⚔️ Sfida Uomo vs Agente (Split-Screen) { run: "auto" }
# @markdown Inserisci il **timestamp** del modello che vuoi affrontare (es. `20261116_101530`), oppure **lascia vuoto** `""` per affrontare l'ultimo modello addestrato:
timestamp_agente = ""  # @param {type:"string"}
scenario_sfida = "campagna_standard"  # @param ["campagna_standard", "deserto_dune", "montagna_estrema", "luna_bassa_gravita", "autostrada_veloce"]

# Il seed è casuale ogni volta per una sfida sempre nuova!
# Se timestamp_agente è vuoto, usa automaticamente 'risultati' oppure l'ultimo modello salvato
modello_scelto = timestamp_agente.strip() if timestamp_agente.strip() else (globals().get("risultati") or None)
play2learn_hillclimb.gioca_vs_agente(
    model_or_result=modello_scelto,
    mission=scenario_sfida,
)
